In [1]:
#!pip install -q sentence-transformers faiss-gpu transformers accelerate

In [2]:
import pandas as pd
import numpy as np
import faiss, glob, kagglehub,torch
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/datasets/mbanaei/all-paraphs-parsed-expanded/data-00002-of-00004.arrow
/kaggle/input/datasets/mbanaei/all-paraphs-parsed-expanded/state.json
/kaggle/input/datasets/mbanaei/all-paraphs-parsed-expanded/data-00003-of-00004.arrow
/kaggle/input/datasets/mbanaei/all-paraphs-parsed-expanded/dataset_info.json
/kaggle/input/datasets/mbanaei/all-paraphs-parsed-expanded/data-00000-of-00004.arrow
/kaggle/input/datasets/mbanaei/all-paraphs-parsed-expanded/data-00001-of-00004.arrow
/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/data-00001-of-00003.arrow
/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/state.json
/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/dataset_info.json
/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/data-00002-of-00003.arrow
/kaggle/i

# Load competition data

In [3]:
train = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test  = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')

option_cols = ['A', 'B', 'C', 'D', 'E']

# Wikipedia Dump

In [4]:
path1 = kagglehub.dataset_download("mbanaei/all-paraphs-parsed-expanded")
path2 = kagglehub.dataset_download("mbanaei/stem-wiki-cohere-no-emb")
print("Path to dataset files:\n", path1,"\n", path2)

Path to dataset files:
 /kaggle/input/datasets/mbanaei/all-paraphs-parsed-expanded 
 /kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb


In [5]:
# after adding the dataset via "Add Input" in Kaggle, inspect what's there
wiki_files = glob.glob('/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/*')
print(wiki_files)

['/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/data-00001-of-00003.arrow', '/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/state.json', '/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/dataset_info.json', '/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/data-00002-of-00003.arrow', '/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb/data-00000-of-00003.arrow']


In [6]:
data_dir = '/kaggle/input/datasets/mbanaei/stem-wiki-cohere-no-emb'
ds = load_dataset('arrow', data_files=f'{data_dir}/data-*.arrow', split='train')
wiki = ds.to_pandas()

print(wiki.columns.tolist())  # confirm the actual text column name
wiki = wiki.dropna(subset=['text']).reset_index(drop=True)  # adjust 'text' if the real column differs

Generating train split: 0 examples [00:00, ? examples/s]

['id', 'title', 'text', 'url', 'wiki_id', 'views', 'paragraph_id', 'langs']


In [7]:
# For a first pass, subsample if the dump is huge (e.g. millions of rows) —
# full-scale indexing can take hours on Kaggle's session limits.
wiki = wiki.sample(n=5000, random_state=42).reset_index(drop=True)

embedder = SentenceTransformer('all-MiniLM-L6-v2', device='cuda' if torch.cuda.is_available() else 'cpu')

def embed_texts(texts, batch_size=256):
    return embedder.encode(
        texts,
        batch_size=batch_size,
        convert_to_numpy=True,
        show_progress_bar=True,
        normalize_embeddings=True  # cosine sim via inner product
    ).astype('float32')

print("Embedding Wikipedia passages (this is the slow part)...")
wiki_embeddings = embed_texts(wiki['text'].tolist())

dimension = wiki_embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(wiki_embeddings)
print(f"Indexed {index.ntotal} passages, dim={dimension}")

# Persist so that its doesn't need to be redone on kernel restart
faiss.write_index(index, "wiki.index")
wiki.to_parquet("wiki_meta.parquet")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Wikipedia passages (this is the slow part)...


Batches:   0%|          | 0/20 [00:00<?, ?it/s]

Indexed 5000 passages, dim=384


# Retrieval function

In [8]:
def retrieve(query, k=3):
    q_vec = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True).astype('float32')
    scores, indices = index.search(q_vec, k)
    return [wiki['text'].iloc[i] for i in indices[0]]

# Load Phi-3-mini

In [9]:
model_name = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
local_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="cuda" if torch.cuda.is_available() else "cpu",
)

config.json:   0%|          | 0.00/967 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/181 [00:00<?, ?B/s]

# Build prompt & call LLM to rank options

In [ ]:
def build_prompt(row, passages):
    context = "\n\n".join(f"- {p}" for p in passages)
    options_text = "\n".join(f"{c}. {row[c]}" for c in option_cols)
    return f"""Use the context below to answer the multiple choice question.

Context:
{context}

Question:
{row['prompt']}

Options:
{options_text}

Respond with ONLY the ranked letters of the three most likely correct options,
best first, separated by commas. Example: "B, D, A"
Answer:"""

def get_llm_ranking(row, k_passages=3, max_new_tokens=10):
    passages = retrieve(row['prompt'], k=k_passages)
    prompt = build_prompt(row, passages)

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(local_llm.device)

    with torch.no_grad():
        output = local_llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(output[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

    letters = []
    for ch in generated.upper():
        if ch in option_cols and ch not in letters:
            letters.append(ch)
    for c in option_cols:
        if len(letters) >= 3:
            break
        if c not in letters:
            letters.append(c)
    return letters[:3]

# Evaluation

In [11]:
def run_inference(df):
    predictions = []
    for _, row in df.iterrows():
        ranking = get_llm_ranking(row)
        predictions.append(ranking)
    return predictions

# --- Evaluate on a held-out slice of train, since test has no labels ---
eval_df = train.sample(n=200, random_state=42).reset_index(drop=True)  # subset for speed; use full train if feasible
eval_preds = run_inference(eval_df)

# Metrics: accuracy, F1, MAP@3

In [12]:
top1_preds = [p[0] for p in eval_preds]
y_true = eval_df['answer'].tolist()

acc = accuracy_score(y_true, top1_preds)
f1 = f1_score(y_true, top1_preds, average='macro')

def map_at_3(y_true, preds_list):
    scores = []
    for true, preds in zip(y_true, preds_list):
        score = 0.0
        for i, p in enumerate(preds[:3]):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return np.mean(scores)

map3 = map_at_3(y_true, eval_preds)

print(f"Accuracy: {acc:.4f}")
print(f"Macro F1: {f1:.4f}")
print(f"MAP@3:    {map3:.4f}")

Accuracy: 0.6100
Macro F1: 0.5996
MAP@3:    0.7625


In [24]:
# run inference on the test set (this is the slow part)
test_preds = run_inference(test)

submission_df = pd.DataFrame({
    "ID": test["id"],  # confirm this matches test.columns.tolist()
    "Prediction": [' '.join(p) for p in test_preds]
})

assert len(submission_df) == len(test), "Row count mismatch!"
assert submission_df.columns.tolist() == ['ID', 'Prediction'], "Column mismatch!"

submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())

   ID Prediction
0   1      A E D
1   2      B A D
2   3      B C E
3   4      E A B
4   5      A C D
